# User Risk Profiling — EDA

Análisis exploratorio de los tres datasets del challenge de seguridad de datos.

**Datasets:**
- `user_inventory.csv` — 500 usuarios
- `permission_inventory.csv` — 2,944 permisos
- `access_logs.csv` — 20,495 eventos de acceso

## 🔑 Hallazgos clave (resumen ejecutivo)

> Síntesis de las señales de riesgo más fuertes detectadas en el EDA. El detalle,
> la evidencia y los gráficos están en las secciones 4–5 (señales directas) y 7
> (señales avanzadas). Cada señal se formaliza luego como una hipótesis (H1–H13).

| Señal | Magnitud | Hipótesis |
|---|---|---|
| 🔴 Usuarios **Inactive con accesos** | 3 usuarios · 53 eventos de acceso | H2 |
| 🔴 **Accesos sin permiso asignado** | 161 accesos · 4 usuarios · **todos** a recursos HIGH/VERY_HIGH | H1 |
| 🟠 **Accesos con permiso expirado** | 93 accesos · 2 usuarios | H5 |
| 🟠 **Volumen anómalo vs. peers** (z>2) | 9 usuarios — USR0020/USR0021 con ~9× la mediana de su grupo (Fintech/Reps) | H3 |
| 🟠 **DELETE/EXPORT por externos** | 1.400 acciones destructivas / de exfiltración | H4 |
| 🟠 **Externos con VERY_HIGH sin expiración** | 20 usuarios · 24 permisos | H7 |
| 🟡 **Privilege escalation** | 2 usuarios acceden a criticidad mayor que sus permisos | H8 |
| 🟡 **Permission bloat** (z>2) | 11 usuarios sobre-aprovisionados vs. sus pares | H12 |
| 🟡 **Cross-department access** | 73 accesos a tipos de recurso fuera del perfil del depto. | H13 |

**Conclusión:** de los 500 usuarios, la enorme mayoría es benigna (**487 sin ninguna
flag**); el riesgo se concentra en un puñado de cuentas con señales claras y
**combinables**. Ese es exactamente el caso de uso del *risk scoring*: ordenar esas
pocas cuentas por severidad en lugar de revisarlas a mano. Los gráficos de este
notebook son **interactivos** (Plotly) — pasá el mouse para ver detalles y zoom.

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

DATA_RAW = Path('../data/01_raw')

# ── Plotly: gráficos interactivos (se ejecutan en local: Jupyter / VS Code) ───
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"
CRIT_COLORS = {"LOW": "#55A868", "MEDIUM": "#4C72B0", "HIGH": "#DD8452", "VERY_HIGH": "#C44E52"}
TYPE_COLORS = ["#4C72B0", "#DD8452"]


In [2]:
users = pd.read_csv(DATA_RAW / 'user_inventory.csv', parse_dates=['created_at'])
perms = pd.read_csv(DATA_RAW / 'permission_inventory.csv', parse_dates=['assigned_at', 'expires_at'])
logs  = pd.read_csv(DATA_RAW / 'access_logs.csv', parse_dates=['timestamp'])

# Orden nominal para criticidad
CRIT_ORDER = ['LOW', 'MEDIUM', 'HIGH', 'VERY_HIGH']
CRIT_DTYPE = pd.CategoricalDtype(categories=CRIT_ORDER, ordered=True)
perms['criticality'] = perms['criticality'].astype(CRIT_DTYPE)

REFERENCE_DATE = logs['timestamp'].max()
print(f'Fecha de referencia (último evento): {REFERENCE_DATE}')
print(f'users: {users.shape} | perms: {perms.shape} | logs: {logs.shape}')

Fecha de referencia (último evento): 2026-05-26 19:53:01
users: (500, 7) | perms: (2944, 6) | logs: (20495, 7)


---
## 1. Calidad de datos

### 1.1 Valores nulos

In [3]:
def nulls_summary(df, name):
    s = df.isnull().sum()
    s = s[s > 0]
    if s.empty:
        print(f'{name}: sin nulos')
        return
    pct = (s / len(df) * 100).round(1)
    print(f'\n{name}:')
    print(pd.DataFrame({'nulos': s, '%': pct}).to_string())

nulls_summary(users, 'user_inventory')
nulls_summary(perms, 'permission_inventory')
nulls_summary(logs,  'access_logs')


user_inventory:
            nulos     %
manager_id     50  10.0

permission_inventory:
            nulos     %
expires_at   2004  68.1
access_logs: sin nulos


### 1.2 Duplicados

In [4]:
for df, name in [(users, 'user_inventory'), (perms, 'permission_inventory'), (logs, 'access_logs')]:
    dups = df.duplicated().sum()
    print(f'{name}: {dups} filas duplicadas ({dups/len(df)*100:.1f}%)')

user_inventory: 0 filas duplicadas (0.0%)
permission_inventory: 0 filas duplicadas (0.0%)
access_logs: 0 filas duplicadas (0.0%)


### 1.3 Integridad referencial

In [5]:
known_users = set(users['user_id'])
known_resources = set(perms['resource_id'])

logs_unknown_users = logs[~logs['user_id'].isin(known_users)]
logs_unknown_resources = logs[~logs['resource_id'].isin(known_resources)]
perms_unknown_users = perms[~perms['user_id'].isin(known_users)]

print(f'Accesos de usuarios no registrados en user_inventory: {len(logs_unknown_users)} ({logs_unknown_users["user_id"].nunique()} únicos)')
print(f'Accesos a recursos sin permiso registrado:            {len(logs_unknown_resources)} ({logs_unknown_resources["resource_id"].nunique()} únicos)')
print(f'Permisos de usuarios no registrados en user_inventory: {len(perms_unknown_users)}')

Accesos de usuarios no registrados en user_inventory: 0 (0 únicos)
Accesos a recursos sin permiso registrado:            0 (0 únicos)
Permisos de usuarios no registrados en user_inventory: 0


### 1.4 Rangos temporales

In [6]:
print('=== user_inventory.created_at ===')
print(f'  min: {users["created_at"].min()}  max: {users["created_at"].max()}')

print('\n=== permission_inventory ===')
print(f'  assigned_at min: {perms["assigned_at"].min()}  max: {perms["assigned_at"].max()}')
print(f'  expires_at  min: {perms["expires_at"].min()}  max: {perms["expires_at"].max()}')

print('\n=== access_logs.timestamp ===')
print(f'  min: {logs["timestamp"].min()}  max: {logs["timestamp"].max()}')

# Permisos donde expires_at < assigned_at (inconsistencia)
bad_dates = perms.dropna(subset=['expires_at'])
bad_dates = bad_dates[bad_dates['expires_at'] < bad_dates['assigned_at']]
print(f'\nPermisos con expires_at < assigned_at (inconsistencia): {len(bad_dates)}')

=== user_inventory.created_at ===
  min: 2024-11-02 00:00:00  max: 2025-10-31 00:00:00

=== permission_inventory ===
  assigned_at min: 2024-11-03 00:00:00  max: 2025-11-30 00:00:00
  expires_at  min: 2025-12-01 00:00:00  max: 2026-11-22 00:00:00

=== access_logs.timestamp ===
  min: 2025-11-01 08:00:01  max: 2026-05-26 19:53:01

Permisos con expires_at < assigned_at (inconsistencia): 0


---
## 2. User Inventory

### 2.1 Tipo de usuario y estado

In [7]:
# Tipo de usuario
fig = px.bar(users["user_type"].value_counts().reset_index(), x="user_type", y="count",
             title="Tipo de usuario", color="user_type",
             color_discrete_sequence=TYPE_COLORS, template="plotly_white")
fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="Usuarios"); fig.show()

# Estado de la cuenta
fig = px.bar(users["status"].value_counts().reset_index(), x="status", y="count",
             title="Estado de la cuenta", color="status",
             color_discrete_sequence=["#55A868", "#C44E52"], template="plotly_white")
fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="Usuarios"); fig.show()

# Tipo × Estado
ct = (pd.crosstab(users["user_type"], users["status"]).reset_index()
        .melt(id_vars="user_type", var_name="status", value_name="count"))
fig = px.bar(ct, x="user_type", y="count", color="status", barmode="group",
             title="Tipo × Estado", color_discrete_map={"Active": "#55A868", "Inactive": "#C44E52"},
             template="plotly_white")
fig.update_layout(xaxis_title="", yaxis_title="Usuarios", legend_title=""); fig.show()

print(pd.crosstab(users["user_type"], users["status"], margins=True))

status     Active  Inactive  All
user_type                       
External       80         0   80
Internal      417         3  420
All           497         3  500


### 2.2 Departamento y rol

In [8]:
# Usuarios por departamento
fig = px.bar(users["department"].value_counts().sort_values().reset_index(),
             x="count", y="department", orientation="h", title="Usuarios por departamento",
             color_discrete_sequence=["#4C72B0"], template="plotly_white")
fig.update_layout(xaxis_title="Cantidad", yaxis_title=""); fig.show()

# Usuarios por rol
fig = px.bar(users["role"].value_counts().sort_values().reset_index(),
             x="count", y="role", orientation="h", title="Usuarios por rol",
             color_discrete_sequence=["#DD8452"], template="plotly_white")
fig.update_layout(xaxis_title="Cantidad", yaxis_title=""); fig.show()

# Heatmap Departamento × Rol
ct_dept_role = pd.crosstab(users["department"], users["role"])
fig = px.imshow(ct_dept_role, text_auto=True, aspect="auto", color_continuous_scale="Blues",
                title="Distribución Departamento × Rol", template="plotly_white")
fig.update_layout(xaxis_title="Rol", yaxis_title="Departamento"); fig.show()

### 2.3 Antigüedad de cuentas

In [9]:
users["account_age_days"] = (REFERENCE_DATE - users["created_at"]).dt.days

# Histograma de antigüedad de cuentas
fig = px.histogram(users, x="account_age_days", nbins=30, title="Antigüedad de cuentas (días)",
                   color_discrete_sequence=["#4C72B0"], template="plotly_white")
fig.update_layout(xaxis_title="Días desde creación", yaxis_title="Usuarios", bargap=0.05); fig.show()

# Antigüedad por tipo de usuario
fig = px.box(users, x="user_type", y="account_age_days", color="user_type",
             title="Antigüedad por tipo de usuario", template="plotly_white",
             color_discrete_sequence=TYPE_COLORS)
fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="Días"); fig.show()

print(users.groupby("user_type")["account_age_days"].describe().round(1))

           count   mean    std    min    25%    50%    75%    max
user_type                                                        
External    80.0  384.1   97.5  219.0  302.8  385.5  468.0  557.0
Internal   420.0  397.4  106.1  207.0  305.2  407.5  488.0  570.0


### 2.4 Usuarios sin manager (posibles cuentas huérfanas)

In [10]:
no_manager = users[users['manager_id'].isna()]
print(f'Usuarios sin manager: {len(no_manager)} ({len(no_manager)/len(users)*100:.1f}%)')
print('Breakdown por tipo y rol:')
print(no_manager.groupby(['user_type', 'role']).size().reset_index(name='count').to_string(index=False))

Usuarios sin manager: 50 (10.0%)
Breakdown por tipo y rol:
user_type       role  count
 External       Reps     11
 Internal      Admin      5
 Internal    Analyst     12
 Internal   Engineer      7
 Internal    Manager      7
 Internal Specialist      8


---
## 3. Permission Inventory

### 3.1 Permisos por usuario

In [11]:
perms_per_user = perms.groupby("user_id").size()

# Distribución de permisos por usuario
fig = px.histogram(perms_per_user.rename("permisos").reset_index(), x="permisos", nbins=30,
                   title="Distribución de permisos por usuario",
                   color_discrete_sequence=["#4C72B0"], template="plotly_white")
fig.update_layout(xaxis_title="Cantidad de permisos", yaxis_title="Usuarios", bargap=0.05); fig.show()

# Top 15 usuarios con más permisos
top_n = perms_per_user.nlargest(15).sort_values()
fig = px.bar(x=top_n.values, y=top_n.index, orientation="h", title="Top 15 usuarios con más permisos",
             color=top_n.values, color_continuous_scale="Oranges", template="plotly_white")
fig.update_layout(xaxis_title="Permisos", yaxis_title="", coloraxis_showscale=False); fig.show()

print(perms_per_user.describe().round(1))

count    500.0
mean       5.9
std        2.3
min        2.0
25%        4.0
50%        6.0
75%        7.0
max       12.0
dtype: float64


### 3.2 Distribución por tipo de recurso y criticidad

In [12]:
# Permisos por tipo de recurso
fig = px.bar(perms["resource_type"].value_counts().sort_values().reset_index(),
             x="count", y="resource_type", orientation="h", title="Permisos por tipo de recurso",
             color_discrete_sequence=["#4C72B0"], template="plotly_white")
fig.update_layout(xaxis_title="Cantidad", yaxis_title=""); fig.show()

# Permisos por criticidad
crit_counts = perms["criticality"].value_counts().reindex(CRIT_ORDER).reset_index()
fig = px.bar(crit_counts, x="criticality", y="count", title="Permisos por criticidad",
             color="criticality", color_discrete_map=CRIT_COLORS,
             category_orders={"criticality": CRIT_ORDER}, template="plotly_white")
fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="Permisos"); fig.show()

# Criticidad por tipo de recurso (apilado)
ct = (pd.crosstab(perms["resource_type"], perms["criticality"])[CRIT_ORDER].reset_index()
        .melt(id_vars="resource_type", var_name="criticality", value_name="count"))
fig = px.bar(ct, x="resource_type", y="count", color="criticality", title="Criticidad por tipo de recurso",
             color_discrete_map=CRIT_COLORS, category_orders={"criticality": CRIT_ORDER},
             template="plotly_white")
fig.update_layout(barmode="stack", xaxis_title="", yaxis_title="Permisos", legend_title="Criticidad"); fig.show()

### 3.3 Permisos expirados

In [13]:
perms_with_exp = perms.dropna(subset=['expires_at'])
expired = perms_with_exp[perms_with_exp['expires_at'] < REFERENCE_DATE]
no_exp = perms[perms['expires_at'].isna()]

print(f'Total permisos:                  {len(perms):>6}')
print(f'Sin fecha de expiración (null):  {len(no_exp):>6} ({len(no_exp)/len(perms)*100:.1f}%)')
print(f'Con expiración futura:           {len(perms_with_exp) - len(expired):>6}')
print(f'EXPIRADOS (expires_at < hoy):    {len(expired):>6} ({len(expired)/len(perms)*100:.1f}%)')

print('\nExpirados por criticidad:')
print(expired['criticality'].value_counts().reindex(CRIT_ORDER).dropna())

print('\nExpirados por tipo de recurso:')
print(expired['resource_type'].value_counts())

Total permisos:                    2944
Sin fecha de expiración (null):    2004 (68.1%)
Con expiración futura:              926
EXPIRADOS (expires_at < hoy):        14 (0.5%)

Expirados por criticidad:
criticality
LOW          2
MEDIUM       2
HIGH         6
VERY_HIGH    4
Name: count, dtype: int64

Expirados por tipo de recurso:
resource_type
api_internal      4
vdi               3
reporting_tool    2
database          2
admin_panel       1
drive             1
email_system      1
Name: count, dtype: int64


### 3.4 Usuarios inactivos con permisos activos (señal de riesgo)

In [14]:
inactive_users = set(users[users['status'] == 'Inactive']['user_id'])
active_perms = perms[(perms['expires_at'].isna()) | (perms['expires_at'] >= REFERENCE_DATE)]

inactive_with_active_perms = active_perms[active_perms['user_id'].isin(inactive_users)]

print(f'Usuarios inactivos:                             {len(inactive_users)}')
print(f'Usuarios inactivos con permisos activos:        {inactive_with_active_perms["user_id"].nunique()}')
print(f'Total permisos activos en usuarios inactivos:   {len(inactive_with_active_perms)}')
print('\nBreakdown por criticidad:')
print(inactive_with_active_perms['criticality'].value_counts().reindex(CRIT_ORDER).dropna())

Usuarios inactivos:                             3
Usuarios inactivos con permisos activos:        3
Total permisos activos en usuarios inactivos:   15

Breakdown por criticidad:
criticality
LOW          3
MEDIUM       4
HIGH         4
VERY_HIGH    4
Name: count, dtype: int64


---
## 4. Access Logs

### 4.1 Volumen de accesos por usuario

In [15]:
accesses_per_user = logs.groupby("user_id").size()

# Distribución de accesos por usuario
fig = px.histogram(accesses_per_user.rename("accesos").reset_index(), x="accesos", nbins=40,
                   title="Distribución de accesos por usuario",
                   color_discrete_sequence=["#4C72B0"], template="plotly_white")
fig.update_layout(xaxis_title="Número de accesos", yaxis_title="Usuarios", bargap=0.05); fig.show()

# Top 15 usuarios por volumen de accesos
top_users = accesses_per_user.nlargest(15).sort_values()
fig = px.bar(x=top_users.values, y=top_users.index, orientation="h",
             title="Top 15 usuarios por volumen de accesos",
             color=top_users.values, color_continuous_scale="Reds", template="plotly_white")
fig.update_layout(xaxis_title="Accesos", yaxis_title="", coloraxis_showscale=False); fig.show()

print(accesses_per_user.describe().round(1))
users_without_logs = known_users - set(logs["user_id"])
print(f"\nUsuarios sin ningún acceso en logs: {len(users_without_logs)}")

count    500.0
mean      41.0
std       30.3
min        8.0
25%       29.0
50%       41.0
75%       50.0
max      396.0
dtype: float64

Usuarios sin ningún acceso en logs: 0


### 4.2 Distribución de acciones y recursos

In [16]:
# Acciones realizadas
fig = px.bar(logs["action"].value_counts().reset_index(), x="action", y="count",
             title="Acciones realizadas", color_discrete_sequence=["#4C72B0"], template="plotly_white")
fig.update_layout(xaxis_title="", yaxis_title="Eventos"); fig.show()

# Recursos accedidos
fig = px.bar(logs["resource_type"].value_counts().sort_values().reset_index(),
             x="count", y="resource_type", orientation="h", title="Recursos accedidos",
             color_discrete_sequence=["#DD8452"], template="plotly_white")
fig.update_layout(xaxis_title="Cantidad", yaxis_title=""); fig.show()

### 4.3 Session duration — outliers

In [17]:
# Distribución de la duración de sesión (lineal y log)
fig = px.histogram(logs, x="session_duration_sec", nbins=50, title="Distribución session_duration_sec",
                   color_discrete_sequence=["#4C72B0"], template="plotly_white")
fig.update_layout(xaxis_title="Segundos", yaxis_title="Eventos", bargap=0.02); fig.show()

fig = px.histogram(logs, x="session_duration_sec", nbins=50, title="Distribución (escala log Y)",
                   color_discrete_sequence=["#4C72B0"], template="plotly_white", log_y=True)
fig.update_layout(xaxis_title="Segundos", yaxis_title="Eventos (log)", bargap=0.02); fig.show()

# Duración por acción
fig = px.box(logs, x="action", y="session_duration_sec", title="Duración por acción",
             color_discrete_sequence=["#4C72B0"], template="plotly_white")
fig.update_layout(xaxis_title="", yaxis_title="Segundos"); fig.show()

print(logs["session_duration_sec"].describe().round(1))
p99 = logs["session_duration_sec"].quantile(0.99)
outliers = logs[logs["session_duration_sec"] > p99]
print(f"\nOutliers (> p99 = {p99:.0f}s): {len(outliers)} ({len(outliers)/len(logs)*100:.2f}%)")

count    20495.0
mean      1814.9
std       1031.3
min         30.0
25%        921.0
50%       1826.0
75%       2705.0
max       3600.0
Name: session_duration_sec, dtype: float64

Outliers (> p99 = 3562s): 203 (0.99%)


### 4.4 Patrones temporales

In [18]:
logs["hour"] = logs["timestamp"].dt.hour
logs["day_of_week"] = logs["timestamp"].dt.day_name()
logs["week"] = logs["timestamp"].dt.to_period("W")

DOW_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Accesos por hora — resaltando el horario nocturno (00-05h)
by_hour = logs.groupby("hour").size().reset_index(name="accesos")
by_hour["franja"] = by_hour["hour"].between(0, 5).map({True: "Fuera de horario (00-05h)", False: "Horario normal"})
fig = px.bar(by_hour, x="hour", y="accesos", color="franja", title="Accesos por hora del día",
             color_discrete_map={"Fuera de horario (00-05h)": "#C44E52", "Horario normal": "#4C72B0"},
             template="plotly_white")
fig.update_layout(xaxis_title="Hora", yaxis_title="Accesos", legend_title=""); fig.show()

# Accesos por día de la semana
by_dow = logs.groupby("day_of_week").size().reindex(DOW_ORDER).reset_index(name="accesos")
fig = px.bar(by_dow, x="day_of_week", y="accesos", title="Accesos por día de la semana",
             color_discrete_sequence=["#DD8452"], template="plotly_white")
fig.update_layout(xaxis_title="", yaxis_title="Accesos"); fig.show()

# Tendencia semanal
weekly = logs.groupby("week").size()
weekly.index = weekly.index.astype(str)
fig = px.line(x=weekly.index, y=weekly.values, markers=True, title="Volumen de accesos por semana",
              template="plotly_white")
fig.update_traces(line_color="#4C72B0")
fig.update_layout(xaxis_title="Semana", yaxis_title="Accesos"); fig.show()

### 4.5 Accesos en horario fuera de oficina (00h–06h)

In [19]:
after_hours = logs[logs['hour'].between(0, 5)]
print(f'Accesos fuera de horario (00-05h): {len(after_hours)} ({len(after_hours)/len(logs)*100:.1f}%)')
print(f'Usuarios distintos: {after_hours["user_id"].nunique()}')

# ¿Son internos o externos?
ah_users = after_hours[['user_id']].drop_duplicates().merge(users[['user_id', 'user_type', 'department']], on='user_id', how='left')
print('\nTipo de usuario en accesos fuera de horario:')
print(ah_users['user_type'].value_counts())

# Accesos en fin de semana
weekend = logs[logs['day_of_week'].isin(['Saturday', 'Sunday'])]
print(f'\nAccesos en fin de semana: {len(weekend)} ({len(weekend)/len(logs)*100:.1f}%)')

Accesos fuera de horario (00-05h): 127 (0.6%)
Usuarios distintos: 7

Tipo de usuario en accesos fuera de horario:
user_type
Internal    7
Name: count, dtype: int64

Accesos en fin de semana: 5968 (29.1%)


### 4.6 Accesos de usuarios inactivos

In [20]:
inactive_logs = logs[logs['user_id'].isin(inactive_users)]
print(f'Accesos de usuarios INACTIVOS: {len(inactive_logs)} ({len(inactive_logs)/len(logs)*100:.1f}%)')
print(f'Usuarios inactivos con accesos: {inactive_logs["user_id"].nunique()} de {len(inactive_users)} inactivos')

if len(inactive_logs) > 0:
    print('\nAcciones realizadas por inactivos:')
    print(inactive_logs['action'].value_counts())
    print('\nRecursos accedidos por inactivos:')
    print(inactive_logs['resource_type'].value_counts())

Accesos de usuarios INACTIVOS: 53 (0.3%)
Usuarios inactivos con accesos: 3 de 3 inactivos

Acciones realizadas por inactivos:
action
QUERY     13
EXPORT    11
LOGIN      9
READ       9
DELETE     6
WRITE      5
Name: count, dtype: int64

Recursos accedidos por inactivos:
resource_type
admin_panel       13
api_internal      12
reporting_tool     9
email_system       7
database           7
drive              3
vdi                2
Name: count, dtype: int64


---
## 5. Análisis cruzado — Señales de riesgo

### 5.1 Accesos sin permiso asignado

In [21]:
# Conjunto de (user_id, resource_id) con permiso asignado
perm_pairs = set(zip(perms['user_id'], perms['resource_id']))
logs['has_permission'] = logs.apply(lambda r: (r['user_id'], r['resource_id']) in perm_pairs, axis=1)

no_perm_logs = logs[~logs['has_permission']]
print(f'Accesos SIN permiso asignado: {len(no_perm_logs)} ({len(no_perm_logs)/len(logs)*100:.1f}%)')
print(f'Usuarios distintos:           {no_perm_logs["user_id"].nunique()}')

# Top usuarios por accesos sin permiso
top_noperm = no_perm_logs.groupby('user_id').size().nlargest(10)
print('\nTop 10 usuarios por accesos sin permiso:')
print(top_noperm.to_string())

Accesos SIN permiso asignado: 161 (0.8%)
Usuarios distintos:           4

Top 10 usuarios por accesos sin permiso:
user_id
USR0080    84
USR0060    27
USR0040    25
USR0041    25


### 5.2 Accesos sin permiso a recursos de alta criticidad

In [22]:
# Mapa resource_id → criticidad máxima asignada
resource_crit = perms.groupby('resource_id')['criticality'].max()
logs['resource_criticality'] = logs['resource_id'].map(resource_crit)

# Re-filtrar desde logs para incluir la nueva columna
no_perm_logs = logs[~logs['has_permission']]
high_crit_no_perm = no_perm_logs[no_perm_logs['resource_criticality'].isin(['HIGH', 'VERY_HIGH'])]
print(f'Accesos sin permiso a recursos HIGH/VERY_HIGH: {len(high_crit_no_perm)}')
print(f'Usuarios afectados: {high_crit_no_perm["user_id"].nunique()}')

print('\nPor criticidad:')
print(high_crit_no_perm['resource_criticality'].value_counts())

print('\nPor tipo de recurso:')
print(high_crit_no_perm['resource_type'].value_counts())

Accesos sin permiso a recursos HIGH/VERY_HIGH: 161


Usuarios afectados: 4

Por criticidad:
resource_criticality
HIGH         93
VERY_HIGH    68
LOW           0
MEDIUM        0
Name: count, dtype: int64

Por tipo de recurso:
resource_type
api_internal      84
vdi               29
payment_portal    25
admin_panel       14
database           9
Name: count, dtype: int64


### 5.3 Accesos con permiso expirado

In [23]:
# Cruce: acceso ocurrió después de que el permiso expiró
perms_exp = perms.dropna(subset=['expires_at'])[['user_id', 'resource_id', 'expires_at', 'criticality']]
merged = logs.merge(perms_exp, on=['user_id', 'resource_id'], how='inner')
expired_access = merged[merged['timestamp'] > merged['expires_at']]

print(f'Accesos realizados con permiso ya expirado: {len(expired_access)}')
print(f'Usuarios distintos: {expired_access["user_id"].nunique()}')
print('\nPor criticidad del permiso:')
print(expired_access['criticality'].value_counts())

Accesos realizados con permiso ya expirado: 93
Usuarios distintos: 2

Por criticidad del permiso:
criticality
HIGH         47
VERY_HIGH    20
LOW          13
MEDIUM       13
Name: count, dtype: int64


### 5.4 Volumen de accesos vs. peer group (departamento × rol)

In [24]:
user_access_count = logs.groupby('user_id').size().reset_index(name='access_count')
user_access_count = user_access_count.merge(users[['user_id', 'department', 'role', 'user_type']], on='user_id', how='left')

peer_stats = user_access_count.groupby(['department', 'role'])['access_count'].agg(
    peer_median='median', peer_std='std', peer_count='count'
).reset_index()

user_access_count = user_access_count.merge(peer_stats, on=['department', 'role'])
user_access_count['z_score'] = (
    (user_access_count['access_count'] - user_access_count['peer_median']) /
    user_access_count['peer_std'].replace(0, np.nan)
)

high_volume = user_access_count[user_access_count['z_score'] > 2].sort_values('z_score', ascending=False)
print(f'Usuarios con volumen anómalo vs. peers (z > 2): {len(high_volume)}')
print(high_volume[['user_id', 'department', 'role', 'access_count', 'peer_median', 'z_score']].head(15).to_string(index=False))


# Scatter interactivo: cada punto es un usuario. Los outliers (z>2) en rojo; el
# hover revela el user_id para saber a quién investigar. La línea punteada marca
# "usuario = mediana de su peer group".
uac = user_access_count.copy()
uac["anomalo"] = uac["z_score"].gt(2).map({True: "z > 2 (anómalo)", False: "normal"})
fig = px.scatter(uac, x="peer_median", y="access_count", color="anomalo",
                 hover_data=["user_id", "department", "role", "z_score"],
                 title="Volumen de accesos vs. mediana del peer group (departamento × rol)",
                 color_discrete_map={"z > 2 (anómalo)": "#C44E52", "normal": "#4C72B0"},
                 template="plotly_white")
_m = uac["peer_median"].max()
fig.add_shape(type="line", x0=0, y0=0, x1=_m, y1=_m, line=dict(dash="dash", color="gray"))
fig.update_layout(xaxis_title="Mediana de accesos del peer group",
                  yaxis_title="Accesos del usuario", legend_title=""); fig.show()

Usuarios con volumen anómalo vs. peers (z > 2): 9
user_id  department       role  access_count  peer_median  z_score
USR0020     Fintech       Reps           396         43.0 4.216122
USR0021     Fintech       Reps           395         43.0 4.204179
USR0080 Engineering   Engineer           114         53.0 4.080808
USR0070     Fintech    Analyst           295         40.5 3.696231
USR0060          HR    Analyst            34         16.0 2.904050
USR0040       Legal      Admin            42         11.5 2.743925
USR0443    Security Specialist            76         53.0 2.316817
USR0244 Engineering Specialist            71         51.0 2.037336
USR0402     Fintech    Manager            51         36.0 2.024359


### 5.5 Comparativa interno vs. externo

In [25]:
logs_with_type = logs.merge(users[["user_id", "user_type"]], on="user_id", how="left")

# Distribución de acciones: Interno vs. Externo (%)
action_by_type = ((pd.crosstab(logs_with_type["action"], logs_with_type["user_type"], normalize="columns") * 100)
                  .reset_index().melt(id_vars="action", var_name="user_type", value_name="pct"))
fig = px.bar(action_by_type, x="action", y="pct", color="user_type", barmode="group",
             title="Distribución de acciones: Interno vs. Externo (%)",
             color_discrete_sequence=TYPE_COLORS, template="plotly_white")
fig.update_layout(xaxis_title="", yaxis_title="%", legend_title=""); fig.show()

# Recursos accedidos: Interno vs. Externo (%)
resource_by_type = ((pd.crosstab(logs_with_type["resource_type"], logs_with_type["user_type"], normalize="columns") * 100)
                    .reset_index().melt(id_vars="resource_type", var_name="user_type", value_name="pct"))
fig = px.bar(resource_by_type, x="resource_type", y="pct", color="user_type", barmode="group",
             title="Recursos accedidos: Interno vs. Externo (%)",
             color_discrete_sequence=TYPE_COLORS, template="plotly_white")
fig.update_layout(xaxis_title="", yaxis_title="%", legend_title=""); fig.show()

# Acciones de alto riesgo por externos
high_risk_actions = ["DELETE", "EXPORT"]
ext_high_risk = logs_with_type[(logs_with_type["user_type"] == "External") &
                                (logs_with_type["action"].isin(high_risk_actions))]
print(f"Acciones DELETE/EXPORT por usuarios EXTERNOS: {len(ext_high_risk)}")
print(ext_high_risk["action"].value_counts())

Acciones DELETE/EXPORT por usuarios EXTERNOS: 1400
action
EXPORT    716
DELETE    684
Name: count, dtype: int64


### 5.6 Resumen de señales por usuario

In [26]:
# Tabla resumen de flags por usuario para orientar el scoring
all_users = users[['user_id', 'user_type', 'department', 'role', 'status']].copy()

# Flag 1: es usuario inactivo
all_users['is_inactive'] = all_users['status'] == 'Inactive'

# Flag 2: tiene accesos siendo inactivo
inactive_log_users = set(inactive_logs['user_id'])
all_users['inactive_with_accesses'] = all_users['user_id'].isin(inactive_log_users)

# Flag 3: accesos sin permiso
no_perm_users = no_perm_logs.groupby('user_id').size().reset_index(name='accesses_without_perm')
all_users = all_users.merge(no_perm_users, on='user_id', how='left').fillna({'accesses_without_perm': 0})

# Flag 4: accesos sin permiso a HIGH/VERY_HIGH
high_no_perm = high_crit_no_perm.groupby('user_id').size().reset_index(name='high_crit_no_perm')
all_users = all_users.merge(high_no_perm, on='user_id', how='left').fillna({'high_crit_no_perm': 0})

# Flag 5: volumen anómalo vs. peers
all_users = all_users.merge(user_access_count[['user_id', 'access_count', 'z_score']], on='user_id', how='left')
all_users['volume_anomaly'] = all_users['z_score'] > 2

# Conteo de flags activos
flag_cols = ['inactive_with_accesses', 'volume_anomaly']
all_users['flags_count'] = (
    (all_users['accesses_without_perm'] > 0).astype(int) +
    (all_users['high_crit_no_perm'] > 0).astype(int) +
    all_users['inactive_with_accesses'].astype(int) +
    all_users['volume_anomaly'].astype(int)
)

print('Distribución de flags por usuario:')
print(all_users['flags_count'].value_counts().sort_index())

print('\nTop 20 usuarios con más flags:')
top_flagged = all_users.sort_values(['flags_count', 'high_crit_no_perm'], ascending=False).head(20)
print(top_flagged[['user_id', 'user_type', 'department', 'role', 'status',
                    'accesses_without_perm', 'high_crit_no_perm', 'volume_anomaly',
                    'inactive_with_accesses', 'flags_count']].to_string(index=False))

Distribución de flags por usuario:
flags_count
0    487
1      9
2      1
3      3
Name: count, dtype: int64

Top 20 usuarios con más flags:
user_id user_type  department       role   status  accesses_without_perm  high_crit_no_perm  volume_anomaly  inactive_with_accesses  flags_count
USR0080  Internal Engineering   Engineer   Active                   84.0               84.0            True                   False            3
USR0060  Internal          HR    Analyst   Active                   27.0               27.0            True                   False            3
USR0040  Internal       Legal      Admin   Active                   25.0               25.0            True                   False            3
USR0041  Internal     Fintech Specialist   Active                   25.0               25.0           False                   False            2
USR0010  Internal    Security    Analyst Inactive                    0.0                0.0           False                    True   

---
## 6. Hipótesis documentadas

Basadas en el EDA anterior. Cada hipótesis está mapeada a tácticas de **MITRE ATT&CK for Enterprise** y controles de **NIST SP 800-53** para justificar su inclusión en el modelo de scoring.

---

### H1 — Acceso sin permiso asignado
Un usuario que accede a recursos para los que no tiene permiso registrado es una anomalía directa. Si esos recursos son `HIGH` o `VERY_HIGH`, el riesgo escala significativamente.

**Evidencia:** 161 accesos de 4 usuarios (USR0080, USR0060, USR0040, USR0041) a recursos `api_internal`, `vdi`, `payment_portal`, `admin_panel`, `database`.

> 🔴 **MITRE ATT&CK:** [T1078 – Valid Accounts](https://attack.mitre.org/techniques/T1078/) — Uso de credenciales para acceder a sistemas sin autorización formal.
> **NIST SP 800-53:** AC-3 (Access Enforcement), AC-6 (Least Privilege).

---

### H2 — Usuarios inactivos con accesos registrados
Un usuario con `status = Inactive` no debería poder acceder a ningún recurso. Cualquier acceso de este tipo es una señal roja independientemente del recurso.

**Evidencia:** 53 accesos de 3 usuarios inactivos. Realizan acciones de alto impacto: QUERY (13), EXPORT (11), DELETE (6). Acceden a `admin_panel` y `api_internal`.

> 🔴 **MITRE ATT&CK:** [T1078.001 – Default Accounts](https://attack.mitre.org/techniques/T1078/001/) / [T1078 – Valid Accounts](https://attack.mitre.org/techniques/T1078/) — Reutilización de cuentas deshabilitadas o comprometidas.
> **NIST SP 800-53:** AC-2 (Account Management) — Las cuentas inactivas deben ser deshabilitadas y sus sesiones terminadas.

---

### H3 — Volumen de accesos anómalo vs. peer group
Usuarios con un volumen de accesos 2+ desviaciones estándar por encima de su grupo de peers (mismo departamento y rol) merecen revisión. Puede indicar exfiltración de datos o cuenta comprometida.

**Evidencia:** 9 usuarios con z > 2. USR0020 y USR0021 (External, Fintech Reps) tienen ~395 accesos frente a una mediana de 43 en su peer group (z ≈ 4.2).

> 🟠 **MITRE ATT&CK:** [T1119 – Automated Collection](https://attack.mitre.org/techniques/T1119/) — Recolección masiva de datos de forma automatizada.
> [T1074 – Data Staged](https://attack.mitre.org/techniques/T1074/) — Preparación de datos para exfiltración posterior.

---

### H4 — Acciones destructivas o de exportación por usuarios externos
Usuarios externos realizando `DELETE` o `EXPORT` sobre recursos de alta criticidad es un patrón anómalo. Los externos deberían tener accesos más restringidos.

**Evidencia:** USR0021 realiza 82 acciones EXPORT y 59 DELETE. USR0020 realiza 70 EXPORT y 61 DELETE — ambos usuarios externos con permisos mínimos (3 y 5 respectivamente).

> 🔴 **MITRE ATT&CK:** [T1485 – Data Destruction](https://attack.mitre.org/techniques/T1485/) (DELETE), [T1030 – Data Transfer Size Limits](https://attack.mitre.org/techniques/T1030/) (EXPORT masivo).
> **NIST SP 800-53:** AC-6 (Least Privilege) — Los externos no deberían tener capacidad de DELETE sobre recursos críticos.

---

### H5 — Permisos expirados utilizados para acceder
Si el sistema no revoca automáticamente permisos vencidos, un atacante o insider podría aprovecharlo. Los accesos post-expiración son señal de control de acceso deficiente.

**Evidencia:** 93 accesos de 2 usuarios con permisos ya expirados. 47 accesos a recursos `HIGH` y 20 a `VERY_HIGH`.

> 🟠 **MITRE ATT&CK:** [T1078 – Valid Accounts](https://attack.mitre.org/techniques/T1078/) — Uso de credenciales/tokens que el sistema debería haber invalidado.
> **NIST SP 800-53:** IA-4 (Identifier Management), AC-2(3) — Revisión automática de cuentas y revocación de privilegios expirados.

---

### H6 — Accesos fuera de horario laboral (00h–05h)
Para usuarios internos, acceder a recursos críticos en horario nocturno es inusual. Para externos, puede ser normal dependiendo de la zona horaria, pero requiere contexto adicional.

**Evidencia:** 127 accesos fuera de horario (0.6% del total), 7 usuarios — todos internos. Señal débil en volumen pero significativa en combinación con otras señales.

> 🟡 **MITRE ATT&CK:** [T1078 – Valid Accounts](https://attack.mitre.org/techniques/T1078/) — Los atacantes operan fuera del horario laboral para reducir detección.
> **Gartner UEBA:** El horario anómalo es una dimensión estándar del behavioral baseline por usuario.

---

### H7 — Usuario externo con permiso VERY_HIGH de duración indefinida
Un usuario externo con permiso permanente (sin `expires_at`) sobre recursos de criticidad muy alta es una configuración de riesgo estructural por diseño, independientemente del comportamiento observado.

**Evidencia:** 24 permisos VERY_HIGH sin vencimiento asignados a 20 usuarios externos. Recursos: `payment_portal` (11), `admin_panel` (8), `vdi` (5).

> 🔴 **MITRE ATT&CK:** [T1098 – Account Manipulation](https://attack.mitre.org/techniques/T1098/) — Persistencia mediante permisos de larga duración que no expiran.
> **NIST SP 800-53:** AC-2(9) – Los accesos temporales a recursos críticos deben tener fecha de expiración obligatoria.

In [27]:
# Validación H7 — externos con permiso VERY_HIGH sin expiración
ext_vh_no_exp = perms[
    (perms['user_id'].isin(users[users['user_type'] == 'External']['user_id'])) &
    (perms['criticality'] == 'VERY_HIGH') &
    (perms['expires_at'].isna())
]
print(f'Usuarios externos con permiso VERY_HIGH sin expiración:')
print(f'  Permisos: {len(ext_vh_no_exp)} | Usuarios: {ext_vh_no_exp["user_id"].nunique()}')
print(ext_vh_no_exp['resource_type'].value_counts())

Usuarios externos con permiso VERY_HIGH sin expiración:
  Permisos: 24 | Usuarios: 20
resource_type
payment_portal    11
admin_panel        8
vdi                5
Name: count, dtype: int64


---
## 7. Hipótesis adicionales — señales avanzadas

Estas hipótesis surgen de cruzar las tres tablas y aplicar las dimensiones del framework UEBA: *volumetría*, *contexto de permisos*, *perfil de acciones* y *topología de acceso*.

---

### H8 — Privilege Escalation: acceso a criticidad superior a los permisos asignados

Un usuario puede tener permiso sobre un recurso de criticidad MEDIUM pero acceder a ese mismo recurso cuando éste tiene asignada una criticidad VERY_HIGH en la tabla de permisos. Esto ocurre cuando un recurso es compartido entre usuarios con distintos niveles, y el sistema no segmenta por nivel dentro del mismo `resource_id`.

Más relevante: si la criticidad máxima de todos los recursos que *accede* supera la criticidad máxima de todos los recursos que tiene *autorizados*, hay una escalada real.

> 🔴 **MITRE ATT&CK:** [T1078 – Valid Accounts](https://attack.mitre.org/techniques/T1078/) + [T1548 – Abuse Elevation Control Mechanism](https://attack.mitre.org/techniques/T1548/)
> El usuario opera en un nivel de privilegio mayor al que su identidad tiene formalmente asignado.
> **NIST SP 800-53:** AC-6 (Least Privilege), AC-3 (Access Enforcement).

In [28]:
# H8 — Privilege Escalation: crit_max_acceso > crit_max_permiso
CRIT_NUM = {'LOW': 1, 'MEDIUM': 2, 'HIGH': 3, 'VERY_HIGH': 4}

# .astype(str) necesario si criticality fue convertida a Categorical anteriormente
user_max_perm = (
    perms.assign(crit_num=perms['criticality'].astype(str).map(CRIT_NUM))
    .groupby('user_id')['crit_num'].max()
    .reset_index(name='max_perm_crit')
)

resource_crit_map = perms.groupby('resource_id')['criticality'].first().reset_index()
resource_crit_map['crit_num'] = resource_crit_map['criticality'].astype(str).map(CRIT_NUM)

logs_crit = logs.merge(resource_crit_map[['resource_id', 'crit_num']], on='resource_id', how='left')
user_max_access = (
    logs_crit.groupby('user_id')['crit_num'].max()
    .reset_index(name='max_access_crit')
)

priv_esc = user_max_perm.merge(user_max_access, on='user_id')
priv_esc['crit_gap'] = priv_esc['max_access_crit'] - priv_esc['max_perm_crit']

affected = priv_esc[priv_esc['crit_gap'] > 0].sort_values('crit_gap', ascending=False)
INV_CRIT = {v: k for k, v in CRIT_NUM.items()}
affected['max_perm_label']   = affected['max_perm_crit'].map(INV_CRIT)
affected['max_access_label'] = affected['max_access_crit'].map(INV_CRIT)

print(f'Usuarios con privilege escalation (acceden a crit > permisos): {len(affected)}')
print(affected[['user_id', 'max_perm_label', 'max_access_label', 'crit_gap']].to_string(index=False))

Usuarios con privilege escalation (acceden a crit > permisos): 2
user_id max_perm_label max_access_label  crit_gap
USR0040         MEDIUM        VERY_HIGH         2
USR0060         MEDIUM        VERY_HIGH         2


### H9 — Discovery / Reconocimiento lateral: recursos únicos anómalos vs. peers

Un atacante con credenciales robadas explora el entorno para mapear qué recursos existen y a cuáles puede llegar. La firma de este patrón es una amplitud de acceso (recursos *distintos*) inusualmente alta comparada con el peer group, aunque el volumen total de accesos sea moderado.

> 🟠 **MITRE ATT&CK:** [T1083 – File and Directory Discovery](https://attack.mitre.org/techniques/T1083/), [T1046 – Network Service Discovery](https://attack.mitre.org/techniques/T1046/)
> La enumeración de recursos es el paso previo a la exfiltración o el movimiento lateral.
> **Gartner UEBA:** La *breadth* de acceso es una dimensión independiente de la volumetría total.

In [29]:
# H9 — Lateral Discovery: distinct_resources anómalo vs. peer group
user_distinct = (
    logs.groupby('user_id')['resource_id']
    .nunique()
    .reset_index(name='distinct_resources')
    .merge(users[['user_id', 'department', 'role']], on='user_id', how='left')
)

peer_dist = (
    user_distinct.groupby(['department', 'role'])['distinct_resources']
    .agg(peer_med='median', peer_std='std')
    .reset_index()
)

user_distinct = user_distinct.merge(peer_dist, on=['department', 'role'])
user_distinct['z_distinct'] = (
    (user_distinct['distinct_resources'] - user_distinct['peer_med'])
    / user_distinct['peer_std'].replace(0, np.nan)
)

h9_flagged = user_distinct[user_distinct['z_distinct'] > 2].sort_values('z_distinct', ascending=False)
print(f'Usuarios con reconocimiento lateral anómalo (z > 2): {len(h9_flagged)}')
print(h9_flagged[['user_id', 'department', 'role', 'distinct_resources', 'peer_med', 'z_distinct']].to_string(index=False))

Usuarios con reconocimiento lateral anómalo (z > 2): 9
user_id  department       role  distinct_resources  peer_med  z_distinct
USR0296     Fintech    Analyst                   8       4.0    2.596294
USR0041     Fintech Specialist                  10       5.5    2.473602
USR0470        Data      Admin                  12       7.0    2.430126
USR0060          HR    Analyst                   8       4.0    2.360004
USR0145 Engineering    Manager                  12       7.5    2.336238
USR0255 Engineering      Admin                  12       8.0    2.183566
USR0040       Legal      Admin                  10       6.0    2.080505
USR0013 Engineering       Reps                   5       3.0    2.019324
USR0437    Security Specialist                   7       5.0    2.018100


### H10 — Exfiltración por ratio de acciones (Data Staging)

Un usuario legítimo tiene un mix balanceado de acciones (READ, WRITE, LOGIN, QUERY). Cuando la proporción de acciones de *extracción* (EXPORT + QUERY) supera significativamente la norma del peer group, es la firma de *data staging* previo a exfiltración.

> 🟠 **MITRE ATT&CK:** [T1074 – Data Staged](https://attack.mitre.org/techniques/T1074/), [T1030 – Data Transfer Size Limits](https://attack.mitre.org/techniques/T1030/)
> El atacante consolida datos de múltiples fuentes antes de exfiltrarlos, generando un ratio anómalo de acciones de lectura/consulta masiva.
> **NIST SP 800-53:** AU-6 (Audit Review, Analysis, and Reporting) — este patrón debe generar alerta automática.

In [30]:
# H10 — Exfiltración: ratio (EXPORT + QUERY) / total vs. peer group
action_counts = (
    logs.groupby(['user_id', 'action']).size()
    .unstack(fill_value=0)
    .reset_index()
)
for col in ['EXPORT', 'QUERY', 'DELETE', 'READ', 'WRITE', 'LOGIN']:
    if col not in action_counts.columns:
        action_counts[col] = 0

action_counts['total_actions'] = action_counts.drop('user_id', axis=1).sum(axis=1)
action_counts['exfil_ratio'] = (
    (action_counts['EXPORT'] + action_counts['QUERY']) / action_counts['total_actions']
)

action_counts = action_counts.merge(users[['user_id', 'department', 'role']], on='user_id', how='left')
peer_exfil = (
    action_counts.groupby(['department', 'role'])['exfil_ratio']
    .agg(peer_med='median', peer_std='std')
    .reset_index()
)
action_counts = action_counts.merge(peer_exfil, on=['department', 'role'])
action_counts['z_exfil'] = (
    (action_counts['exfil_ratio'] - action_counts['peer_med'])
    / action_counts['peer_std'].replace(0, np.nan)
)

h10_flagged = action_counts[action_counts['z_exfil'] > 2].sort_values('z_exfil', ascending=False)
print(f'Usuarios con ratio de exfiltración anómalo (z > 2): {len(h10_flagged)}')
cols = ['user_id', 'department', 'role', 'EXPORT', 'QUERY', 'total_actions', 'exfil_ratio', 'z_exfil']
print(h10_flagged[cols].to_string(index=False))

Usuarios con ratio de exfiltración anómalo (z > 2): 7
user_id  department       role  EXPORT  QUERY  total_actions  exfil_ratio  z_exfil
USR0093     Fintech       Reps       8      9             31     0.548387 2.733774
USR0009       Legal Specialist       5      2             10     0.700000 2.423248
USR0198       Legal   Engineer       5      2             10     0.700000 2.247011
USR0316     Fintech    Manager       6     12             34     0.529412 2.193377
USR0216 Engineering    Analyst      12     13             58     0.431034 2.145575
USR0195       Legal    Manager       1      5              9     0.666667 2.131919
USR0035 Engineering Specialist      17      6             46     0.500000 2.113434


### H11 — Externo con comportamiento de insider comprometido

Un usuario externo debería tener un perfil de acceso restringido y especializado. Cuando un externo combina (a) volumen de accesos comparable a un empleado interno, (b) distribución uniforme entre todos los tipos de acción, y (c) acceso masivo a recursos sensibles con pocos permisos formales, es la firma de una credencial comprometida o una herramienta automatizada operando bajo identidad externa.

La distribución de acciones verdaderamente plana (sin tipo dominante) es particularmente anómala: los usuarios legítimos tienen patrones especializados según su función.

> 🔴 **MITRE ATT&CK:** [T1078.004 – Valid Accounts: Cloud Accounts](https://attack.mitre.org/techniques/T1078/004/), [T1119 – Automated Collection](https://attack.mitre.org/techniques/T1119/)
> Una herramienta automatizada ejecutando comandos bajo credenciales externas robadas generaría exactamente este patrón.
> **Gartner UEBA:** La *entropía de acciones* (uniformidad de la distribución) es un feature de detección de bots.

In [31]:
# H11 — Externo con comportamiento de insider: entropía de acciones + volumen
from scipy.stats import entropy

external_users = set(users[users['user_type'] == 'External']['user_id'])

def action_entropy(group):
    counts = group['action'].value_counts()
    probs = counts / counts.sum()
    return entropy(probs, base=2)  # máximo teórico = log2(6) ≈ 2.58 para 6 acciones distintas

user_entropy = (
    logs[logs['user_id'].isin(external_users)]
    .groupby('user_id')
    .apply(action_entropy, include_groups=False)
    .reset_index(name='action_entropy')
)

ext_volume = (
    logs[logs['user_id'].isin(external_users)]
    .groupby('user_id').size()
    .reset_index(name='total_accesses')
)
ext_perm_count = (
    perms[perms['user_id'].isin(external_users)]
    .groupby('user_id').size()
    .reset_index(name='perm_count')
)

ext_profile = (
    user_entropy
    .merge(ext_volume, on='user_id')
    .merge(ext_perm_count, on='user_id', how='left')
    .merge(users[['user_id', 'department', 'role', 'status']], on='user_id')
)
ext_profile['perm_count'] = ext_profile['perm_count'].fillna(0)
ext_profile['accesses_per_perm'] = ext_profile['total_accesses'] / ext_profile['perm_count'].replace(0, np.nan)

# Señal combinada: entropía alta (> 2.3) Y volumen alto (> percentil 90 de externos)
vol_p90 = ext_profile['total_accesses'].quantile(0.90)
h11_flagged = ext_profile[
    (ext_profile['action_entropy'] > 2.3) &
    (ext_profile['total_accesses'] > vol_p90)
].sort_values('action_entropy', ascending=False)

print(f'Externos con comportamiento de insider (alta entropía + alto volumen): {len(h11_flagged)}')
print(h11_flagged[['user_id', 'department', 'role', 'total_accesses', 'perm_count',
                    'accesses_per_perm', 'action_entropy']].to_string(index=False))

print('\nDetalle de distribución de acciones para los casos más críticos:')
for uid in h11_flagged.head(3)['user_id']:
    dist = logs[logs['user_id'] == uid]['action'].value_counts()
    print(f'  {uid}: {dist.to_dict()}')

Externos con comportamiento de insider (alta entropía + alto volumen): 8
user_id  department role  total_accesses  perm_count  accesses_per_perm  action_entropy
USR0020     Fintech Reps             396           5          79.200000        2.579272
USR0113 Engineering Reps              68           4          17.000000        2.574037
USR0021     Fintech Reps             395           3         131.666667        2.568295
USR0013 Engineering Reps              71           5          14.200000        2.555878
USR0029    Security Reps              75           2          37.500000        2.549550
USR0395 Engineering Reps              70           2          35.000000        2.544917
USR0326 Engineering Reps              67           4          16.750000        2.509615
USR0003 Engineering Reps              68           2          34.000000        2.501005

Detalle de distribución de acciones para los casos más críticos:
  USR0020: {'LOGIN': 72, 'WRITE': 71, 'EXPORT': 70, 'QUERY': 66, 'DEL

### H12 — Permission Bloat: violación del principio de Least Privilege

Un usuario que acumula significativamente más permisos que sus pares de mismo rol y departamento amplía su superficie de ataque. Si esa cuenta se compromete, el blast radius es mayor. Este es un riesgo de *configuración*, no de comportamiento observable, pero contribuye al score de riesgo estructural.

> 🟡 **MITRE ATT&CK:** [T1098 – Account Manipulation](https://attack.mitre.org/techniques/T1098/) — La acumulación de permisos puede ser el resultado de manipulación de cuentas.
> **NIST SP 800-53:** AC-6 (Least Privilege) — "Organizations employ the principle of least privilege, allowing only authorized accesses for users which are necessary to accomplish assigned tasks."
> **NIST SP 800-53:** AC-2(7) – Revisión periódica de cuentas privilegiadas.

In [32]:
# H12 — Permission Bloat vs. peer group
perm_count_user = perms.groupby('user_id').size().reset_index(name='perm_count')
perm_count_user = perm_count_user.merge(
    users[['user_id', 'department', 'role', 'user_type']], on='user_id', how='left'
)

peer_perm = (
    perm_count_user.groupby(['department', 'role'])['perm_count']
    .agg(peer_med='median', peer_std='std')
    .reset_index()
)
perm_count_user = perm_count_user.merge(peer_perm, on=['department', 'role'])
perm_count_user['z_perms'] = (
    (perm_count_user['perm_count'] - perm_count_user['peer_med'])
    / perm_count_user['peer_std'].replace(0, np.nan)
)

h12_flagged = perm_count_user[perm_count_user['z_perms'] > 2].sort_values('z_perms', ascending=False)
print(f'Usuarios con permission bloat (z > 2): {len(h12_flagged)}')
print(h12_flagged[['user_id', 'user_type', 'department', 'role',
                    'perm_count', 'peer_med', 'z_perms']].to_string(index=False))

Usuarios con permission bloat (z > 2): 11
user_id user_type  department       role  perm_count  peer_med  z_perms
USR0296  Internal     Fintech    Analyst           8       4.0 2.596294
USR0082  Internal          HR    Manager          12       7.0 2.492928
USR0145  Internal Engineering    Manager          12       7.5 2.336238
USR0470  Internal        Data      Admin          12       7.0 2.325651
USR0404  Internal  Operations    Manager          11       7.5 2.237452
USR0236  Internal          HR    Analyst           7       4.0 2.224860
USR0255  Internal Engineering      Admin          12       8.0 2.183566
USR0085  Internal       Legal   Engineer           7       5.0 2.057983
USR0134  Internal          HR      Admin          11       7.0 2.034191
USR0013  External Engineering       Reps           5       3.0 2.019324
USR0437  Internal    Security Specialist           7       5.0 2.018100


### H13 — Cross-department resource access: movimiento lateral estructural

Si un usuario accede a un `resource_type` para el cual su departamento entero no tiene permisos asignados, el acceso no puede explicarse por el perfil normal del rol. Es una señal de movimiento lateral: el usuario (o su credencial) está operando fuera de su dominio funcional.

> 🔴 **MITRE ATT&CK:** [T1021 – Remote Services](https://attack.mitre.org/techniques/T1021/) — Acceso lateral a servicios/recursos de otros dominios.
> [T1083 – File and Directory Discovery](https://attack.mitre.org/techniques/T1083/) — Exploración de recursos fuera del alcance habitual.
> **NIST SP 800-53:** AC-4 (Information Flow Enforcement) — Los controles de flujo deben impedir el acceso inter-departamental no autorizado.

In [33]:
# H13 — Cross-department resource access
# Qué resource_types tiene habilitados cada departamento (según permisos)
dept_allowed_rtypes = (
    perms.merge(users[['user_id', 'department']], on='user_id')
    .groupby('department')['resource_type']
    .apply(set)
    .to_dict()
)

logs_dept = logs.merge(users[['user_id', 'department', 'role']], on='user_id', how='left')

def is_cross_dept(row):
    dept = row['department']
    if pd.isna(dept):
        return False
    return row['resource_type'] not in dept_allowed_rtypes.get(dept, set())

logs_dept['cross_dept'] = logs_dept.apply(is_cross_dept, axis=1)
cross_dept_logs = logs_dept[logs_dept['cross_dept']]

print(f'Accesos a resource_type fuera del perfil del departamento: {len(cross_dept_logs)}')
print(f'Usuarios distintos: {cross_dept_logs["user_id"].nunique()}')

print('\nDetalle por departamento → resource_type accedido:')
detail = (
    cross_dept_logs.groupby(['user_id', 'department', 'role', 'resource_type'])
    .size()
    .reset_index(name='accesses')
    .sort_values('accesses', ascending=False)
)
print(detail.to_string(index=False))

Accesos a resource_type fuera del perfil del departamento: 73
Usuarios distintos: 3

Detalle por departamento → resource_type accedido:
user_id department       role  resource_type  accesses
USR0060         HR    Analyst payment_portal        18
USR0040      Legal      Admin            vdi        15
USR0041    Fintech Specialist            vdi        14
USR0060         HR    Analyst       database         9
USR0040      Legal      Admin    admin_panel         7
USR0041    Fintech Specialist    admin_panel         7
USR0040      Legal      Admin payment_portal         3


---
## 8. Resumen consolidado de hipótesis

Tabla de referencia para el diseño del modelo de scoring (Parte 2).

In [34]:
# Tabla resumen de todas las hipótesis
hypotheses = [
    ("H1",  "Acceso sin permiso asignado",                    "T1078",       "🔴 Crítica"),
    ("H2",  "Usuario Inactive con accesos",                   "T1078.001",   "🔴 Crítica"),
    ("H3",  "Volumen anómalo vs. peer group",                 "T1119/T1074", "🟠 Alta"),
    ("H4",  "DELETE/EXPORT masivo por externos",              "T1485/T1030", "🔴 Crítica"),
    ("H5",  "Permiso expirado utilizado",                     "T1078",       "🟠 Alta"),
    ("H6",  "Acceso fuera de horario laboral (00-05h)",       "T1078",       "🟡 Media"),
    ("H7",  "Externo con perm VERY_HIGH sin vencimiento",     "T1098",       "🟠 Alta"),
    ("H8",  "Privilege escalation (crit acceso > permisos)",  "T1078/T1548", "🔴 Crítica"),
    ("H9",  "Discovery: recursos únicos anómalos vs. peers",  "T1083/T1046", "🟠 Alta"),
    ("H10", "Exfiltración: ratio EXPORT+QUERY anómalo",       "T1074/T1030", "🟠 Alta"),
    ("H11", "Externo con entropía de acciones de insider",    "T1078.004",   "🔴 Crítica"),
    ("H12", "Permission bloat (least privilege violation)",   "T1098",       "🟡 Media"),
    ("H13", "Cross-dept resource access (movimiento lateral)","T1021/T1083", "🔴 Crítica"),
]

import pandas as pd
df_hyp = pd.DataFrame(hypotheses, columns=["ID", "Hipótesis", "MITRE ATT&CK", "Severidad"])
print(df_hyp.to_string(index=False))

 ID                                       Hipótesis MITRE ATT&CK Severidad
 H1                     Acceso sin permiso asignado        T1078 🔴 Crítica
 H2                    Usuario Inactive con accesos    T1078.001 🔴 Crítica
 H3                  Volumen anómalo vs. peer group  T1119/T1074    🟠 Alta
 H4               DELETE/EXPORT masivo por externos  T1485/T1030 🔴 Crítica
 H5                      Permiso expirado utilizado        T1078    🟠 Alta
 H6        Acceso fuera de horario laboral (00-05h)        T1078   🟡 Media
 H7      Externo con perm VERY_HIGH sin vencimiento        T1098    🟠 Alta
 H8   Privilege escalation (crit acceso > permisos)  T1078/T1548 🔴 Crítica
 H9   Discovery: recursos únicos anómalos vs. peers  T1083/T1046    🟠 Alta
H10        Exfiltración: ratio EXPORT+QUERY anómalo  T1074/T1030    🟠 Alta
H11     Externo con entropía de acciones de insider    T1078.004 🔴 Crítica
H12    Permission bloat (least privilege violation)        T1098   🟡 Media
H13 Cross-dept resource a